# Qwen3.8-35B-A3B — Colab distill + GGUF

This notebook **clones the repo**, installs it, then **SFT**s the **Qwen3.6-35B-A3B** MoE runtime on [r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation](https://huggingface.co/datasets/r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation) (`sft_balanced`: Qwen3.8-Max-Preview + GLM-5.2 + Kimi K3), optionally **chases open Qwen3.8-27B** with reverse-KL / on-policy KD, then exports Unsloth-style XL GGUFs.

1. `git clone` + `pip install -e ".[colab]"`
2. Hugging Face login + Drive (checkpoints live on Drive under `hq_maxmix`, chase under `hq_27b`)
3. Confirm the 35B-A3B graph
4. **Stage A SFT** on the Max/GLM/Kimi traces (B/C off — the assistant text is already in the dataset)
5. **After A finishes:** Chase 27B (section 5b) — prompts → 27B generate → reverse KL → on-policy GKD
6. Save tokenizer onto the adapter you will merge
7. Merge LoRA and write `UD-Q4_K_XL` / `UD-Q3_K_XL` GGUFs

There is no 64-sample smoke run. Re-run section 5 after a disconnect; completed steps resume. **Do not start section 5b while Stage A is still training.**

Stage A alone is **not** open Qwen3.8-27B KD and will not equal 27B dense. Decode stays ~3B active. Dataset license is `other` — **noncommercial research**.

**Runtime → A100 (40GB+). High-RAM for GGUF.** Hugging Face login is interactive if you have not added a Colab secret named `HF_TOKEN`. Add `GITHUB_TOKEN` only if the repo is private.

Open from GitHub: [Open in Colab](https://colab.research.google.com/github/birdup000/qwen3-8-35b-a3b/blob/main/notebooks/Qwen3.8-35B-A3B_Colab.ipynb)


## 1. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime → Change runtime type → A100 GPU."
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024**3
print(props.name, f"{vram:.1f} GB", "torch", torch.__version__)
if vram < 22:
    raise RuntimeError(f"Need ~22GB+ VRAM for 4-bit HQ distill; this GPU has {vram:.1f} GB.")
print("For GGUF export, also pick a High-RAM runtime (peak ~70GB merged HF + ~67GB BF16 GGUF).")

## 2. Clone the repo and install

This cell is self-contained. You do **not** upload a zip. It clones `birdup000/qwen3-8-35b-a3b` and runs setup.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/birdup000/qwen3-8-35b-a3b.git"
REPO_BRANCH = "main"
DEST = Path("/content/Qwen3.8-35B-A3B")

try:
    from google.colab import userdata
    os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
except Exception:
    pass

token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""
clone_url = REPO_URL
if token and REPO_URL.startswith("https://") and "@" not in REPO_URL[8:]:
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)

def sh(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd)

if (DEST / ".git").is_dir():
    sh(["git", "fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=DEST)
    sh(["git", "checkout", REPO_BRANCH], cwd=DEST)
    sh(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=DEST)
elif not (DEST / "qwen3_8_moe" / "configuration.py").is_file():
    DEST.parent.mkdir(parents=True, exist_ok=True)
    sh(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(DEST)])
else:
    print("Repo files already present")

os.chdir(DEST)
if str(DEST) not in sys.path:
    sys.path.insert(0, str(DEST))

sh([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
sh([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], cwd=DEST)

from scripts.colab_pipeline import try_install_flash_delta
print(try_install_flash_delta())

from qwen3_8_moe import parameter_report, qwen38_35b_a3b_config
text = qwen38_35b_a3b_config().text_config
assert (text.hidden_size, text.num_hidden_layers, text.num_experts) == (2048, 40, 256)
print("Cloned", DEST)
print(f"Graph {text.num_hidden_layers}L / {text.num_experts}E  active~{parameter_report()['active']/1e9:.2f}B")
print(sorted(p.name for p in DEST.iterdir() if not p.name.startswith(".")))

## 3. Hugging Face login + Drive

If you have no Colab secret, this cell opens a Hugging Face login widget. Create a token at https://huggingface.co/settings/tokens (read access is enough). Accept the licenses on [Qwen3.6-35B-A3B](https://huggingface.co/Qwen/Qwen3.6-35B-A3B) and [Qwen3.8-27B](https://huggingface.co/Qwen/Qwen3.8-27B).

To skip the widget next time: Colab left sidebar → 🔑 **Secrets** → add `HF_TOKEN` → enable Notebook access.

In [ ]:
import os
from pathlib import Path
from google.colab import drive
from huggingface_hub import login, notebook_login

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

if token:
    os.environ["HF_TOKEN"] = token
    login(token=token)
    print("Logged in with HF_TOKEN")
else:
    print("No Colab secret named HF_TOKEN.")
    print("Paste a token from https://huggingface.co/settings/tokens in the widget.")
    notebook_login()

drive.mount("/content/drive")
DRIVE_OUT = Path("/content/drive/MyDrive/Qwen3.8-35B-A3B")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
TEACHER_ID = "Qwen/Qwen3.8-27B"
STUDENT_ID = "Qwen/Qwen3.6-35B-A3B"
HQ_DIR = DRIVE_OUT / "hq_maxmix"
HQ_DIR.mkdir(parents=True, exist_ok=True)
print("Drive ready at", DRIVE_OUT)
print("HQ checkpoints", HQ_DIR)

## 4. Confirm the 35B-A3B graph

In [ ]:
from scripts.colab_pipeline import detect_runtime, ensure_repo
from qwen3_8_moe import parameter_report, qwen38_35b_a3b_config

ensure_repo()
print(detect_runtime())
report = parameter_report()
print(f"Total {report['total'] / 1e9:.3f}B   active {report['active'] / 1e9:.3f}B")
print("Serve as", qwen38_35b_a3b_config().to_hf_dict()["architectures"][0])

## 5. Distill Max/GLM/Kimi traces onto 35B-A3B

Run **after** Drive is mounted. Checkpoints live at `DRIVE_OUT/hq_maxmix` (a **new** folder so the UltraChat `hq/` run is not reused). Re-run this cell after a disconnect.

**Stage A** SFT’s the 35B-A3B on [`r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation`](https://huggingface.co/datasets/r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation) (`sft_balanced`): Qwen3.8-Max-Preview + GLM-5.2 + Kimi K3 traces. That is **not** open Qwen3.8-27B. Dataset license is `other` (mixed upstream + Model Studio terms) — treat as **noncommercial research**.

Stage B (27B generate) and Stage C (top-k KD) stay **off** unless you set `HQ_STAGE` to `"b"` / `"c"`. `"all"` with this corpus runs Stage A only.

On a 40GB A100 the packed MoE expert banks stay BF16 (~64GB) and sit on CPU. NF4 Linears stay on the GPU. If an UltraChat Stage A is still running, stop it — that jsonl must not be reused here.


In [ ]:
from pathlib import Path
from scripts.colab_pipeline import hq_progress, run_hq_pipeline, run_hq_stage_a, run_hq_stage_b, run_hq_stage_c

TEACHER_ID = globals().get("TEACHER_ID", "Qwen/Qwen3.8-27B")
STUDENT_ID = globals().get("STUDENT_ID", "Qwen/Qwen3.6-35B-A3B")
DRIVE_OUT = Path(globals().get("DRIVE_OUT", "/content/drive/MyDrive/Qwen3.8-35B-A3B"))
HQ_DIR = Path(globals().get("HQ_DIR", DRIVE_OUT / "hq_maxmix"))
HQ_DIR.mkdir(parents=True, exist_ok=True)

SFT_DATASET = "r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation"
SFT_CONFIG = "sft_balanced"

# "a" | "b" | "c" | "all"  — with this dataset, "all" is Stage A only.
HQ_STAGE = "a"
MAX_ROWS = 16000
STAGE_A_STEPS = 4000  # None = one pass over whatever tokenized
SEQ_LEN = 2048
MAX_TRACES = 5000
STAGE_C_STEPS = 4000

print("HQ work dir", HQ_DIR)
print("SFT", SFT_DATASET, SFT_CONFIG)
print(hq_progress(HQ_DIR, max_rows=MAX_ROWS, max_traces=MAX_TRACES, stage_c_steps=STAGE_C_STEPS))

kwargs = dict(
    student_id=STUDENT_ID,
    work_dir=HQ_DIR,
    seq_len=SEQ_LEN,
    fourbit=True,
    sft_dataset=SFT_DATASET,
    sft_config=SFT_CONFIG,
)
if HQ_STAGE == "a":
    hq_paths = run_hq_stage_a(max_rows=MAX_ROWS, steps=STAGE_A_STEPS, **kwargs)
elif HQ_STAGE == "b":
    hq_paths = run_hq_stage_b(
        teacher_id=TEACHER_ID,
        max_traces=MAX_TRACES,
        max_new_tokens=1024,
        **{k: kwargs[k] for k in ("work_dir", "seq_len", "fourbit")},
    )
elif HQ_STAGE == "c":
    hq_paths = run_hq_stage_c(
        steps=STAGE_C_STEPS,
        **{k: kwargs[k] for k in ("student_id", "work_dir", "seq_len", "fourbit")},
    )
else:
    hq_paths = run_hq_pipeline(
        teacher_id=TEACHER_ID,
        max_rows=MAX_ROWS,
        max_traces=MAX_TRACES,
        stage_a_steps=STAGE_A_STEPS,
        stage_c_steps=STAGE_C_STEPS,
        max_new_tokens=1024,
        **kwargs,
    )
print({key: str(value) for key, value in hq_paths.items()})
print(hq_progress(HQ_DIR, max_rows=MAX_ROWS, max_traces=MAX_TRACES, stage_c_steps=STAGE_C_STEPS))


## 5b. Chase Qwen3.8-27B (after Stage A)

Keep section 5 running. That Max/GLM/Kimi SFT is the warmup bump. When `a_step` hits 4000, run this cell. It will **error on purpose** if Stage A is still short of 4000, so you cannot copy a mid-run LoRA or fight the A100.

Work dir is `hq_27b` (the maxmix adapter is not overwritten). Teacher is **open Qwen3.8-27B**, not Max-Preview.

1. **prompts** — OpenCodeReasoning train contests, OpenThoughts3 questions, Magicoder problems, leftover Stage A users. Answers dropped so 27B writes them.
2. **b** — 27B generate + top-256 logits
3. **c** — truncated reverse KL on those traces (seeds from finished `hq_maxmix/adapter`)
4. **d** — student rollouts scored by 27B (on-policy GKD)

Set `CHASE_STAGE` to `"prompts"`, then `"b"`, then `"c"`, then `"d"`.


In [ ]:
from pathlib import Path
from scripts.colab_pipeline import run_chase_27b
from scripts.hq_distill import hq_progress

DRIVE_OUT = Path(globals().get("DRIVE_OUT", "/content/drive/MyDrive/Qwen3.8-35B-A3B"))
HQ_DIR = Path(globals().get("HQ_DIR", DRIVE_OUT / "hq_maxmix"))
SEED_ADAPTER = HQ_DIR / "adapter"
CHASE_DIR = DRIVE_OUT / "hq_27b"
CHASE_DIR.mkdir(parents=True, exist_ok=True)

# "prompts" | "b" | "c" | "d" | "all"
# Keep section 5 running. Flip this only after a_step hits MIN_A_STEP.
CHASE_STAGE = "prompts"
MIN_A_STEP = 4000
MAX_PROMPTS = 8000
MAX_TRACES = 4000          # 80GB: 20000
MAX_NEW_TOKENS = 1536      # 80GB: 4096
SEQ_LEN = 2048             # 80GB: 4096
STAGE_C_STEPS = 6000
STAGE_D_STEPS = 3000       # additional on-policy steps (separate d_step)

print("Stage A status", hq_progress(HQ_DIR, max_rows=16000, stage_c_steps=STAGE_C_STEPS))
print("seed adapter", SEED_ADAPTER)
print("chase dir", CHASE_DIR)
chase_paths = run_chase_27b(
    work_dir=CHASE_DIR,
    seed_adapter_dir=SEED_ADAPTER,
    stage=CHASE_STAGE,
    max_prompts=MAX_PROMPTS,
    max_traces=MAX_TRACES,
    max_new_tokens=MAX_NEW_TOKENS,
    seq_len=SEQ_LEN,
    stage_c_steps=STAGE_C_STEPS,
    stage_d_steps=STAGE_D_STEPS,
    stage_a_jsonl=HQ_DIR / "stage_a.jsonl",
    min_a_step=MIN_A_STEP,
)
print({key: str(value) for key, value in chase_paths.items()})


## 6. Tokenizer on the HQ adapter

Writes the 3.8 tokenizer files onto the adapter you will merge. Prefers `hq_27b` once chase traces exist, otherwise `hq_maxmix`.


In [ ]:
from pathlib import Path
from transformers import AutoTokenizer

TEACHER_ID = globals().get("TEACHER_ID", "Qwen/Qwen3.8-27B")
DRIVE_OUT = Path(globals().get("DRIVE_OUT", "/content/drive/MyDrive/Qwen3.8-35B-A3B"))
HQ_DIR = Path(globals().get("HQ_DIR", DRIVE_OUT / "hq_maxmix"))
chase_adapter = DRIVE_OUT / "hq_27b" / "adapter"
hq_adapter = HQ_DIR / "adapter"
if (chase_adapter / "adapter_config.json").is_file() and (DRIVE_OUT / "hq_27b" / "traces.jsonl").is_file():
    hq_adapter = chase_adapter
    print("Using chase adapter")
if not (hq_adapter / "adapter_config.json").is_file():
    raise FileNotFoundError(f"No HQ adapter yet. Finish section 5 first. Looked in {hq_adapter}")

tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID, trust_remote_code=True)
tokenizer.save_pretrained(hq_adapter)
print("HQ adapter ready at", hq_adapter)
print(sorted(p.name for p in DRIVE_OUT.iterdir()))
print(sorted(p.name for p in hq_adapter.iterdir())[:20])


## 7. Serve like Qwen3.6

HF / vLLM after a full merge (section 8 also merges for GGUF):

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.6-35B-A3B", device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, "/content/drive/MyDrive/Qwen3.8-35B-A3B/hq_maxmix/adapter").merge_and_unload()
model.save_pretrained("/path/to/Qwen3.8-35B-A3B-merged")
```

```bash
vllm serve /path/to/Qwen3.8-35B-A3B-merged
```

```python
extra_body = {
    "chat_template_kwargs": {
        "enable_thinking": True,
        "preserve_thinking": True,
        "reasoning_effort": "xhigh",
    }
}
```

llama.cpp / Ollama after section 8:

```bash
./llama-cli -m Qwen3.8-35B-A3B-UD-Q4_K_XL.gguf -ngl 99 -ot ".ffn_.*_exps.=CPU"
```


## 8. GGUF — Unsloth XL (`UD-Q4_K_XL` / `UD-Q3_K_XL`)

Run this **after section 5 finishes** (and after 5b if you chased 27B). Prefers `DRIVE_OUT/hq_27b/adapter` once `traces.jsonl` exists, else `hq_maxmix`. Merges into `Qwen/Qwen3.6-35B-A3B`, converts with llama.cpp, then quantizes with the public Unsloth/Bartowski XL recipe (Q8 embeddings + output, Q8 `ssm_out`, higher-bit `ffn_down`).

Default file: `Qwen3.8-35B-A3B-UD-Q4_K_XL.gguf` (~20GB). Set `QUANTS = ["Q3_K_XL"]` or both.

**Disk:** High-RAM runtime. Peak is ~70GB merged HF + ~67GB BF16 GGUF. The cell deletes the 27B teacher cache first. BF16 GGUF is removed after the XL file is written.


In [ ]:
import os
from pathlib import Path

DRIVE_OUT = Path(globals().get("DRIVE_OUT", "/content/drive/MyDrive/Qwen3.8-35B-A3B"))
HQ_DIR = Path(globals().get("HQ_DIR", DRIVE_OUT / "hq_maxmix"))
STUDENT_ID = globals().get("STUDENT_ID", "Qwen/Qwen3.6-35B-A3B")
chase_adapter = DRIVE_OUT / "hq_27b" / "adapter"
LORA_DIR = HQ_DIR / "adapter"
if (chase_adapter / "adapter_config.json").is_file() and (DRIVE_OUT / "hq_27b" / "traces.jsonl").is_file():
    LORA_DIR = chase_adapter
if not (LORA_DIR / "adapter_config.json").is_file():
    raise FileNotFoundError(f"No HQ LoRA adapter. Finish section 5 first. Looked in {LORA_DIR}")
print("Using adapter", LORA_DIR)

# Q4_K_XL ~20GB (recommended). Q3_K_XL ~16GB. Use both if disk allows.
QUANTS = ["Q4_K_XL"]
GGUF_WORK = Path("/content/qwen38_gguf")

# cmake + a compiler for llama-quantize / llama-imatrix
os.system("apt-get update -qq && apt-get install -y -qq build-essential cmake git >/dev/null")

from scripts.colab_pipeline import export_unsloth_xl_gguf

gguf_paths = export_unsloth_xl_gguf(
    work_dir=GGUF_WORK,
    lora_dir=LORA_DIR,
    out_dir=DRIVE_OUT,
    quants=QUANTS,
    base_id=STUDENT_ID,
    free_teacher=True,
    imatrix=True,
    keep_bf16=False,
)
print({key: str(value) for key, value in gguf_paths.items()})
print("GGUF files:", sorted(p.name for p in DRIVE_OUT.glob("*.gguf")))
